# Stage 02: Indexing & Vector Search Playground

A modular, general-purpose sandbox for experimenting with:
1. **Embedding Models** (Remote APIs, zero local PyTorch dependency).
2. **Vector Math & Metrics** (Cosine Similarity, Dot Product, L2 Distance).
3. **Semantic Visualization** (2D PCA projection of arbitrary text clusters).
4. **Vector Store Operations** (FAISS indexing, similarity search, scoring).

---
## 1. Environment & Setup
Loads credentials from `.env` and imports core dependencies.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from dotenv import load_dotenv

# 1. Resolve project root reliably
current = Path.cwd()
project_root = current
for p in [current, current.parent, current.parent.parent]:
    if (p / '.env').exists() or (p / 'pyproject.toml').exists():
        project_root = p
        break

load_dotenv(project_root / '.env')

# 2. Add 02-indexing to path for shared modules
indexing_dir = project_root / '02-indexing'
if str(indexing_dir) not in sys.path:
    sys.path.insert(0, str(indexing_dir))

from huggingface_api import HuggingFaceAPIEmbeddings

print('[OK] Setup completed successfully!')


---
## 2. Embedding Models Lab
Initialize remote embeddings and inspect vector properties.

In [ ]:
# Initialize remote embeddings (runs via Hugging Face Serverless API)
embeddings = HuggingFaceAPIEmbeddings(
    model="sentence-transformers/all-MiniLM-L6-v2",
    normalize_embeddings=True,
    expected_dimension=384
)
print(f"Active Embedding Model: {embeddings}")

# Helper function to inspect any text embedding
def inspect_embedding(text: str):
    vec = embeddings.embed_query(text)
    norm = np.linalg.norm(vec)
    print(f"Text:        '{text}'")
    print(f"Dimension:   {len(vec)}")
    print(f"L2 Norm:     {norm:.4f} (1.0 = Normalized)")
    print(f"Sample:      {vec[:4]}...")
    return vec

sample_vec = inspect_embedding("RAG architectures revolutionize enterprise knowledge search.")

---
## 3. Vector Similarity & Distance Calculator
Calculates Cosine Similarity, Dot Product, and pairwise matrices for arbitrary texts.

In [ ]:
def compute_similarity_matrix(texts: list[str]):
    """Embeds a list of texts and returns the full pairwise cosine similarity matrix."""
    vecs = np.array(embeddings.embed_documents(texts))
    # For normalized vectors, dot product equals cosine similarity
    return np.dot(vecs, vecs.T)

test_sentences = [
    "Deep learning algorithms optimize neural network weights.",
    "Machine learning models learn patterns from training data.",
    "Traditional Italian pizza is baked in a wood-fired brick oven.",
    "Pasta carbonara is cooked with egg yolks, pecorino, and guanciale."
]

matrix = compute_similarity_matrix(test_sentences)
labels = ['DL/Neural', 'ML/Patterns', 'Pizza', 'Pasta']
header = f"{'Category':<12}" + ''.join(f"{lbl:>14}" for lbl in labels)
print('Pairwise Cosine Similarity Matrix:')
print(header)
print('-' * len(header))
for i, row in enumerate(matrix):
    row_str = f"{labels[i]:<12}" + ''.join(f"{val:>14.4f}" for val in row)
    print(row_str)


---
## 4. Interactive 2D Semantic Visualization (PCA)
Projects high-dimensional vectors (384D) into 2D space to observe semantic clustering.

In [ ]:
def plot_semantic_clusters(items: dict[str, str]):
    """Embeds labeled sentences and visualizes them in 2D via PCA."""
    labels = list(items.keys())
    texts = list(items.values())
    
    # Generate embeddings
    vectors = np.array(embeddings.embed_documents(texts))
    
    # PCA via SVD (Pure NumPy, zero external dependencies)
    centered = vectors - np.mean(vectors, axis=0)
    _, _, vt = np.linalg.svd(centered, full_matrices=False)
    coords_2d = np.dot(centered, vt[:2].T)
    
    # Plotting
    plt.figure(figsize=(10, 6))
    for i, label in enumerate(labels):
        x, y = coords_2d[i, 0], coords_2d[i, 1]
        plt.scatter(x, y, s=160, zorder=4)
        plt.annotate(
            f"{label}\n({texts[i][:20]}...)",
            xy=(x, y),
            xytext=(x + 0.02, y + 0.015),
            fontsize=9,
            fontweight="bold"
        )
    
    plt.title("Semantic Clustering in 2D Space (PCA Projection)", fontsize=13, fontweight="bold", pad=12)
    plt.xlabel("Principal Component 1", fontsize=11)
    plt.ylabel("Principal Component 2", fontsize=11)
    plt.grid(True, linestyle="--", alpha=0.5)
    plt.tight_layout()
    plt.show()

playground_corpus = {
    "AI/Neural": "Artificial neural networks process complex high-dimensional information.",
    "AI/Training": "Model training requires gradient descent and loss optimization.",
    "Food/Pizza": "Neapolitan pizza has a blistered crust with melted mozzarella.",
    "Food/Recipe": "Classic lasagna layers rich tomato meat sauce with bechamel.",
    "Sports/Soccer": "The football team secured victory in the championship final.",
    "Sports/Running": "Marathon runners pace their stamina over 42 kilometers."
}

plot_semantic_clusters(playground_corpus)

---
## 5. Vector Store & Retrieval Sandbox (FAISS)
Builds an in-memory vector index and evaluates similarity search.

In [ ]:
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document

# 1. Prepare sample knowledge base
docs = [
    Document(page_content="Vector embeddings represent semantic text in multi-dimensional space.", metadata={"category": "rag", "id": 1}),
    Document(page_content="FAISS enables fast similarity search on dense vector collections.", metadata={"category": "vector_db", "id": 2}),
    Document(page_content="LangChain provides abstractions for chaining LLMs, prompts, and retrievers.", metadata={"category": "framework", "id": 3}),
    Document(page_content="Chunking divides long documents into coherent semantic passages.", metadata={"category": "ingestion", "id": 4})
]

# 2. Build FAISS vector store
vectorstore = FAISS.from_documents(docs, embeddings)
print(f"Indexed {len(docs)} documents in FAISS.")

# 3. Query with similarity scores (Lower distance = higher similarity for L2)
query = "How to store and search embedding vectors quickly?"
results_with_scores = vectorstore.similarity_search_with_score(query, k=2)

print(f"\nQuery: '{query}'")
for rank, (doc, score) in enumerate(results_with_scores, 1):
    print(f"\nMatch #{rank} (Distance Score: {score:.4f}):")
    print(f"  Content:  {doc.page_content}")
    print(f"  Metadata: {doc.metadata}")

---
## Summary
This playground serves as your active testing laboratory for Stage 02.
Feel free to paste your own texts, test new queries, or evaluate different vector store configurations.